In [1]:
from pathlib import Path
import subprocess

checkout = Path("/content/sam-replication")
if not checkout.exists():
    subprocess.run([
        "git", "clone", "--branch", "repro",
        "https://github.com/Siddharth-magesh/sam-replication.git",
        str(checkout)
    ], check=True)

subprocess.run(["nvidia-smi"], check=True)

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
import json
from pathlib import Path

root = Path("/content/sam_l4_compat")

# Avoid starting a second installation in the CLI-managed runtime.
marker = root / "setup_launch.json"
if marker.exists():
    pid = json.loads(marker.read_text())["pid"]
    command = Path(f"/proc/{pid}/cmdline")
    if command.exists() and b"setup_job.py" in command.read_bytes():
        raise RuntimeError(
            "The earlier setup is still running. Wait, then rerun this cell."
        )

notebook = json.loads(
    (checkout / "repro_notebooks" /
     "04_wrn28_10_cifar10_paper_replication.ipynb").read_text()
)
cells = [
    "".join(cell["source"])
    for cell in notebook["cells"]
    if cell["cell_type"] == "code"
]

namespace = {}
exec(next(c for c in cells if "WORK_ROOT = Path.home()" in c), namespace)

namespace.update(
    WORK_ROOT=root,
    ENV_PREFIX=root / "envs/sam",
    PYTHON=root / "envs/sam/bin/python",
    REPO=root / "work/sam",
    SESSION=root / "manual_setup",
    RUN_SETUP=True,
    RUN_TRAINING=False,
)

exec(next(c for c in cells if "def setup_command(" in c), namespace)

Planned session: /root/sam_paper_replication/runs/20261118_041521_504d33
Setup, GPU execution and result rendering default to disabled.
Setup: nvidia-smi -L

COMMAND: ['nvidia-smi', '-L']
GPU 0: NVIDIA L4 (UUID: GPU-e57f3cb4-db16-b8c4-d0eb-b97656794228)
Setup: /content/sam_l4_compat/bin/micromamba --version
COMMAND: ['nvidia-smi', '-L']
GPU 0: NVIDIA L4 (UUID: GPU-e57f3cb4-db16-b8c4-d0eb-b97656794228)

COMMAND: [PosixPath('/content/sam_l4_compat/bin/micromamba'), '--version']
2.10.0
Setup: git clone https://github.com/Siddharth-magesh/sam-replication.git /content/sam_l4_compat/work/sam
COMMAND: [PosixPath('/content/sam_l4_compat/bin/micromamba'), '--version']
2.10.0

COMMAND: ['git', 'clone', 'https://github.com/Siddharth-magesh/sam-replication.git', PosixPath('/content/sam_l4_compat/work/sam')]
Cloning into '/content/sam_l4_compat/work/sam'...
Setup: git -C /content/sam_l4_compat/work/sam checkout --detach f7b9ff46ee0deab8004d5508baf68b7e7c3dad1a
COMMAND: ['git', 'clone', 'https://git

In [6]:
import os
import subprocess
from pathlib import Path

root = Path("/content/sam_l4_compat")
prefix = root / "envs/sam"
python = prefix / "bin/python"
repo = root / "work/sam"
log_dir = root / "manual_setup"

assert python.is_file(), "Complete the setup cell first."
assert repo.is_dir(), "Pinned SAM source checkout is missing."
log_dir.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PATH"] = str(prefix / "bin") + ":" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = (
    str(prefix / "lib") + ":" + env.get("LD_LIBRARY_PATH", "")
)
env["PYTHONPATH"] = str(repo.parent)
env["CUDA_HOME"] = str(prefix)
env["CUDA_PATH"] = str(prefix)
env["CUDA_VISIBLE_DEVICES"] = "0"
env["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=" + str(root / "cuda")
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

code = r"""
import sys
from importlib.metadata import version

import numpy as np
import jax
import jaxlib
import flax

installed_jaxlib = version("jaxlib")
print("Python:", sys.version, flush=True)
print("JAX:", jax.__version__, flush=True)
print("jaxlib module:", jaxlib.__version__, flush=True)
print("jaxlib package:", installed_jaxlib, flush=True)
print("Flax:", flax.__version__, flush=True)

assert sys.version_info[:2] == (3, 8), "Expected Python 3.8."
assert jax.__version__ == "0.2.6", "Unexpected JAX version."
assert jaxlib.__version__.split("+")[0] == "0.1.57", \
    "Unexpected jaxlib module version."
assert installed_jaxlib == "0.1.57+cuda110", \
    "Expected the pinned CUDA 11.0 jaxlib wheel."
assert flax.__version__ == "0.2.2", "Unexpected Flax version."

devices = jax.devices()
print("Devices:", devices, flush=True)
assert len(devices) == 1 and devices[0].platform == "gpu", \
    "Expected exactly one GPU."

from absl import flags
from sam.sam_jax.models import load_model, wide_resnet

flags.FLAGS(["wrn_initialization_check"])
flags.FLAGS.use_additional_skip_connections = False

print("Initializing original WRN-28-10...", flush=True)
module = wide_resnet.WideResnet.partial(
    blocks_per_group=4,
    channel_multiplier=10,
    num_outputs=10,
)
model, state = load_model.create_image_model(
    jax.random.PRNGKey(0), 32, 32, module, 3
)

for index, leaf in enumerate(jax.tree_leaves((model.params, state))):
    if hasattr(leaf, "block_until_ready"):
        leaf.block_until_ready()
    assert np.isfinite(np.asarray(leaf)).all(), \
        "Non-finite model/state leaf: {}".format(index)

parameters = sum(int(x.size) for x in jax.tree_leaves(model.params))
assert parameters > 0, "Model has no parameters."

print("Parameters:", parameters, flush=True)
print("WRN-28-10 initialization: PASS", flush=True)
"""

print("Running initialization check; maximum five minutes.")
result = subprocess.run(
    [
        "timeout", "--kill-after=20s", "300s",
        str(python), "-u", "-c", code,
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

log = log_dir / "wrn_initialization.log"
log.write_text(result.stdout, encoding="utf-8")

print(result.stdout or "(No subprocess output)")
print("Exit code:", result.returncode)
print("Saved log:", log)

if result.returncode == 0:
    print("Initialization passed. SAM/SGD training is not tested yet.")
elif result.returncode == 124:
    print("Reached the five-minute timeout. Send the output above.")
else:
    print("Initialization did not pass. Send the output above.")

Running initialization check; maximum five minutes.
Python: 3.8.20 | packaged by conda-forge | (default, Sep 30 2024, 17:52:49) 
[GCC 13.3.0]
JAX: 0.2.6
jaxlib module: 0.1.57
jaxlib package: 0.1.57+cuda110
Flax: 0.2.2
Devices: [GpuDevice(id=0)]
Initializing original WRN-28-10...
2026-11-18 04:27:56.748679: W external/org_tensorflow/tensorflow/stream_executor/gpu/asm_compiler.cc:235] Your CUDA software stack is old. We fallback to the NVIDIA driver for some compilation. Update your CUDA version to get the best performance. The ptxas error was: ptxas fatal   : Value 'sm_89' is not defined for option 'gpu-name'

2026-11-18 04:27:56.748815: W external/org_tensorflow/tensorflow/compiler/xla/service/gpu/nvptx_compiler.cc:249] Couldn't read CUDA driver version.
2026-11-18 04:27:56.763627: W external/org_tensorflow/tensorflow/stream_executor/gpu/asm_compiler.cc:235] Your CUDA software stack is old. We fallback to the NVIDIA driver for some compilation. Update your CUDA version to get the best 

In [7]:
import json
import os
import subprocess
import time
from pathlib import Path
from datetime import datetime, timezone

root = Path("/content/sam_l4_compat")
prefix = root / "envs/sam"

env = os.environ.copy()
env.update({
    "PATH": str(prefix / "bin") + ":" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": str(prefix / "lib") + ":" + env.get("LD_LIBRARY_PATH", ""),
    "PYTHONPATH": str(root / "work"),
    "CUDA_HOME": str(prefix),
    "CUDA_PATH": str(prefix),
    "CUDA_VISIBLE_DEVICES": "0",
    "XLA_FLAGS": "--xla_gpu_cuda_data_dir=" + str(root / "cuda"),
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
    "TFDS_DATA_DIR": str(root / "tensorflow_datasets"),
    "PYTHONHASHSEED": "0",
})

run = root / (
    "smoke_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
)
run.mkdir(parents=True, exist_ok=False)

worker = r'''
import hashlib
import json
import sys
from functools import partial
from pathlib import Path
from importlib.metadata import version

import numpy as np
import tensorflow as tf
tf.config.experimental.set_visible_devices([], "GPU")

import jax
import jax.numpy as jnp
import flax
from flax import serialization
from absl import flags
from sam.sam_jax.models import load_model, wide_resnet
from sam.sam_jax.datasets import dataset_source
from sam.sam_jax.training_utils import flax_training as official

method, output = sys.argv[1], Path(sys.argv[2])
output.mkdir(parents=True, exist_ok=False)

assert sys.version_info[:2] == (3, 8)
assert jax.__version__ == "0.2.6"
assert version("jaxlib") == "0.1.57+cuda110"
assert flax.__version__ == "0.2.2"
assert len(jax.devices()) == 1 and jax.devices()[0].platform == "gpu"

F = flags.FLAGS
F(["smoke"])
for name, value in dict(
    learning_rate=0.1,
    weight_decay=0.0005,
    gradient_clipping=5.0,
    use_rmsprop=False,
    sam_rho=0.05 if method == "SAM" else 0.0,
    sync_perturbations=False,
    label_smoothing=0.0,
    no_weight_decay_on_bn=False,
    ema_decay=0.0,
    use_test_set=True,
    use_additional_skip_connections=False,
    run_seed=0,
    compute_top_5_error_rate=False,
).items():
    setattr(F, name, value)

np.random.seed(0)
tf.random.set_seed(0)

def sync(tree):
    for x in jax.tree_leaves(tree):
        if hasattr(x, "block_until_ready"):
            x.block_until_ready()
    return tree

def finite(tree):
    return all(np.isfinite(np.asarray(x)).all()
               for x in jax.tree_leaves(tree))

def digest(params):
    return hashlib.sha256(serialization.to_bytes(params)).hexdigest()

def shard(batch):
    return {
        k: np.asarray(v).reshape((8, -1) + tuple(v.shape[1:]))
        for k, v in batch.items()
    }

print(method, "loading CIFAR-10...", flush=True)
data = dataset_source.Cifar10(256, "basic", "none")
assert data.num_training_obs == 50000

print(method, "initializing WRN-28-10...", flush=True)
module = wide_resnet.WideResnet.partial(
    blocks_per_group=4, channel_multiplier=10, num_outputs=10
)
model, state = load_model.create_image_model(
    jax.random.PRNGKey(0), 32, 32, module, 3
)
optimizer = official.create_optimizer(model, 0.0)
states = jax.tree_map(lambda x: jnp.stack([x] * 8), state)

initial_hash = digest(model.params)
initial_bn = [np.asarray(x).copy() for x in jax.tree_leaves(states)]

schedule = official.get_cosine_schedule(200, 0.1, 50000, 256)
step = jax.jit(jax.vmap(
    partial(official.train_step,
            learning_rate_fn=schedule, l2_reg=0.0005),
    in_axes=(None, 0, 0, 0),
    out_axes=(None, 0, 0, None),
    axis_name="batch",
))
evaluate = jax.jit(jax.vmap(
    official.eval_step,
    in_axes=(None, 0, 0),
    axis_name="batch",
))

history = []
for index, batch in enumerate(data.get_train(True).take(10)):
    batch = shard(batch)
    assert batch["image"].shape == (8, 32, 32, 32, 3)
    keys = jax.random.split(
        jax.random.fold_in(jax.random.PRNGKey(0), index), 8
    )
    optimizer, states, metrics, lr = sync(
        step(optimizer, states, batch, keys)
    )
    assert finite(metrics) and finite(lr), "Non-finite training metrics."
    row = dict(
        batch=index + 1,
        loss=float(np.asarray(metrics["train_loss"]).mean()),
        learning_rate=float(np.asarray(lr)),
    )
    history.append(row)
    print(method, row, flush=True)

assert len(history) == 10
assert int(np.asarray(optimizer.state.step)) == 10
assert finite((optimizer, states)), "Non-finite optimizer/BatchNorm state."
assert digest(optimizer.target.params) != initial_hash, "Parameters unchanged."
assert any(
    not np.array_equal(before, np.asarray(after))
    for before, after in zip(initial_bn, jax.tree_leaves(states))
), "BatchNorm state did not update."

test_batch = next(iter(data.get_test()))
count = int(test_batch["image"].shape[0])
assert count % 8 == 0
test_batch = shard(test_batch)
before = sync(evaluate(optimizer.target, states, test_batch))
assert finite(before), "Non-finite evaluation."

checkpoint = output / "checkpoint.msgpack"
checkpoint.write_bytes(serialization.to_bytes(
    dict(optimizer=optimizer, states=states)
))
restored = serialization.from_bytes(
    dict(optimizer=optimizer, states=states),
    checkpoint.read_bytes(),
)

for original, recovered in zip(
    jax.tree_leaves((optimizer, states)),
    jax.tree_leaves((restored["optimizer"], restored["states"])),
):
    np.testing.assert_array_equal(np.asarray(original), np.asarray(recovered))

after = sync(evaluate(
    restored["optimizer"].target, restored["states"], test_batch
))
assert finite(after), "Non-finite restored evaluation."
for key in before:
    np.testing.assert_allclose(
        np.asarray(before[key]), np.asarray(after[key]),
        rtol=1e-5, atol=1e-5,
    )

result = dict(
    method=method,
    status="passed",
    batches=10,
    batch_size=256,
    logical_subbatches=8,
    initialization_sha256=initial_hash,
    parameters_updated=True,
    batchnorm_updated=True,
    checkpoint_restored=True,
    evaluation_examples=count,
    evaluation_loss=float(np.asarray(before["loss"])[0]) / count,
    history=history,
)
(output / "result.json").write_text(json.dumps(result, indent=2))
print(method, "SMOKE PASS", flush=True)
'''

script = run / "smoke_worker.py"
script.write_text(worker)

attempts = []
for method in ("SGD", "SAM"):
    print(f"\nStarting {method}; first compilation can take several minutes.")
    started = time.monotonic()

    # Stream logs into the notebook and preserve them on disk.
    with (run / f"{method}.log").open("w") as log:
        process = subprocess.Popen(
            [
                "timeout", "--signal=INT", "--kill-after=20s", "900s",
                str(prefix / "bin/python"), "-u", str(script),
                method, str(run / method),
            ],
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            exit_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
        finally:
            attempts.append({
                "method": method,
                "wall_seconds": time.monotonic() - started,
                "exit_code": process.poll(),
            })
            (run / "attempts.json").write_text(json.dumps(attempts, indent=2))

    if exit_code != 0:
        raise RuntimeError(
            f"{method} stopped with exit code {exit_code}. "
            f"Send the end of {run / (method + '.log')}. "
            "No further method will run."
        )

results = [
    json.loads((run / method / "result.json").read_text())
    for method in ("SGD", "SAM")
]
assert results[0]["initialization_sha256"] == results[1]["initialization_sha256"]

print("\nBOTH FUNCTIONAL SMOKE TESTS PASSED")
print("Results and logs:", run)
print("Total diagnostic hours:",
      sum(a["wall_seconds"] for a in attempts) / 3600)

Streaming output truncated to the last 5000 lines.

2026-11-18 04:44:31.129459: W external/org_tensorflow/tensorflow/stream_executor/gpu/asm_compiler.cc:235] Your CUDA software stack is old. We fallback to the NVIDIA driver for some compilation. Update your CUDA version to get the best performance. The ptxas error was: ptxas fatal   : Value 'sm_89' is not defined for option 'gpu-name'

2026-11-18 04:44:31.198336: W external/org_tensorflow/tensorflow/stream_executor/gpu/asm_compiler.cc:235] Your CUDA software stack is old. We fallback to the NVIDIA driver for some compilation. Update your CUDA version to get the best performance. The ptxas error was: ptxas fatal   : Value 'sm_89' is not defined for option 'gpu-name'

2026-11-18 04:44:31.272620: W external/org_tensorflow/tensorflow/stream_executor/gpu/asm_compiler.cc:235] Your CUDA software stack is old. We fallback to the NVIDIA driver for some compilation. Update your CUDA version to get the best performance. The ptxas error was: ptxas

In [8]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

root = Path("/content/sam_l4_compat")
backup = Path("/content/drive/MyDrive/sam_replication")
backup.mkdir(parents=True, exist_ok=True)

# 1. Preserve results, logs, checkpoints and diagnostic timing.
run_name = "smoke_20261118_043333_653758"
source = root / run_name
assert source.is_dir(), f"Results folder missing: {source}"

destination = backup / "runs" / run_name
shutil.copytree(source, destination, dirs_exist_ok=True)

# Verify that every source file was copied with the same size.
for file in source.rglob("*"):
    if file.is_file():
        copied = destination / file.relative_to(source)
        assert copied.is_file()
        assert copied.stat().st_size == file.stat().st_size

print("Results saved:", destination)

# 2. Preserve the prepared CIFAR-10 dataset and download cache.
data_source = root / "tensorflow_datasets"
data_destination = backup / "tensorflow_datasets"
assert data_source.is_dir(), f"Dataset cache missing: {data_source}"

shutil.copytree(data_source, data_destination, dirs_exist_ok=True)

for file in data_source.rglob("*"):
    if file.is_file():
        copied = data_destination / file.relative_to(data_source)
        assert copied.is_file()
        assert copied.stat().st_size == file.stat().st_size

print("Dataset saved:", data_destination)
print("BACKUP COMPLETE — copied files verified by size.")

Mounted at /content/drive
Results saved: /content/drive/MyDrive/sam_replication/runs/smoke_20261118_043333_653758
Dataset saved: /content/drive/MyDrive/sam_replication/tensorflow_datasets
BACKUP COMPLETE — copied files verified by size.


In [9]:
import json
import subprocess
from pathlib import Path

run_name = "smoke_20261118_043333_653758"
locations = [
    Path("/content/sam_l4_compat") / run_name / "attempts.json",
    Path("/content/drive/MyDrive/sam_replication/runs")
    / run_name / "attempts.json",
]

record = next((p for p in locations if p.is_file()), None)
assert record is not None, "Mount Drive or check the results folder."

attempts = json.loads(record.read_text())
total_seconds = 0.0

print("RECORDED SMOKE-TEST TIME")
print(f"{'Method':<10} {'Minutes':>10} {'Hours':>10} {'Exit code':>12}")

for attempt in attempts:
    seconds = float(attempt["wall_seconds"])
    total_seconds += seconds
    print(
        f"{attempt['method']:<10}"
        f"{seconds / 60:>10.2f}"
        f"{seconds / 3600:>10.4f}"
        f"{str(attempt['exit_code']):>12}"
    )

print(f"\nTotal: {total_seconds / 60:.2f} minutes")
print(f"Elapsed time with one allocated GPU: {total_seconds / 3600:.4f} hours")
print("Excludes earlier setup, initialization tests and idle session time.")

print("\nCURRENT GPU SNAPSHOT — not historical usage")
result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,utilization.gpu,memory.used,memory.total,power.draw",
        "--format=csv",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout if result.returncode == 0 else result.stderr)

RECORDED SMOKE-TEST TIME
Method        Minutes      Hours    Exit code
SGD            13.56    0.2261           0
SAM             5.70    0.0950           0

Total: 19.27 minutes
Elapsed time with one allocated GPU: 0.3211 hours
Excludes earlier setup, initialization tests and idle session time.

CURRENT GPU SNAPSHOT — not historical usage
name, utilization.gpu [%], memory.used [MiB], memory.total [MiB], power.draw [W]
NVIDIA L4, 0 %, 0 MiB, 23034 MiB, 17.09 W

